<a href="https://colab.research.google.com/github/IsaacFigNewton/CommitteeHearingDiscourseParser/blob/main/DH2024_DigitalDemocracy_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Accessing & working with the Digital Democracy Corpus: Comprehensive Proceedings of Four State Legislatures 2015-2018


This notebook will act as a guide for a person with minimal knowledge of coding to make use of the files and retrieve useful information from the corpus.


The dataset can be downloaded directly from this website: https://huggingface.co/datasets/iatpp/digitaldemocracy-2015-2018

This corpus is distributed under the cc-by-nc-sa-4.0 license. Please review before accessing. https://creativecommons.org/licenses/by-nc-sa/4.0/

Please cite as:

Khosmood, F., Dekhtyar, A., Ellwein, S., White, B., "The Digital Democracy Corpus: Comprehensive Proceedings of Four State Legislatures 2015-2018", Digital Humanities, Washington, DC, August 2024.

# Download files

The Digital Democracy Corpus can be downloaded from here: https://huggingface.co/datasets/iatpp/digitaldemocracy-2015-2018

The following code cell will downloaded the relevant files from the data repository and store them in the Colab runtime.

 Files in Colab runtime storage can be accessed by clicking on the folder button on the left sidebar. Double click on a file to view a preview of the file. Anything stored Colab runtime storage will disappear once the session is over. To save these files to your physical computer, right click on the file you'd like to save and click download.

In [1]:
from pathlib import Path
import subprocess
import zipfile
import os

#This is where the unzipped corpus file is stored
CORPUS_FILE_PATH = 'DH2024_Corpus_Release/'
corpus_dir = Path(CORPUS_FILE_PATH)
zip_path = Path("digitaldemocracy-2015-2018/DH2024_Corpus_Release.zip")
repo_dir = Path("digitaldemocracy-2015-2018")

# Clone repository if not present
if not repo_dir.is_dir():
    print("Corpus directory not found. Cloning repository...")
    subprocess.run(
        ["git", "clone", "https://huggingface.co/datasets/iatpp/digitaldemocracy-2015-2018"],
        check=True,
    )
    print("Repository cloned successfully.")

# Extract zip file if corpus directory doesn't exist
if not corpus_dir.is_dir():
    if zip_path.exists():
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('.')
        print("Extraction complete.")
    else:
        print(f"Error: {zip_path} not found!")
else:
    print(f"Corpus already extracted at {corpus_dir}")

Corpus already extracted at DH2024_Corpus_Release


# Read the relevant files into Python Objects by set

In [2]:
import sys
import csv

# Increase CSV field size limit for large bill text (Windows compatible)
try:
    csv.field_size_limit(sys.maxsize)
except OverflowError:
    # Windows workaround
    maxInt = int(2**31 - 1)
    csv.field_size_limit(maxInt)

# Use the updated corpus path
CORPUS_FILE_PATH = 'DH2024_Corpus_Release/'

#We have 4 state sets
VALID_STATES=["CA", "FL", "NY", "TX"]

#Each state set has 9 csv files for each session year
CSV_FILENAMES=['bills','committeeHearings', 'committeeRosters',
               'committees', 'hearings', 'legislature',
               'people','speeches','videos']

#Load data from CSVs into a Python object to reference later
#Input:
#  Required: file_name (type:String) (Ex: speeches, bills, etc)
#  Optional: states (type:List of Strings or None) (Ex: ["CA"], ["FL,TX"])
#     -If not specified (states=None), function returns data for all states
#  Optional: years (type:List of Ints or None) (Ex:[2018], [2015,2016])
#     -If not specified (years=None), function returns data for all valid years
#Output:
#  Payload (Type: Dict) (Ex: {column_headers:['pid','cid','date'], rows:[[0,2,2018],[2,1,2018]]})
def load_content(file_name, states=None, years=None):
  #Only accept valid states, Corpus only contains data on CA, FL, NY, and TX legislations
  if states is not None and not all(item in VALID_STATES for item in states):
    raise Exception("Invalid State Abbv(s), corpus only contains data on CA, FL, NY, and TX")
  #Only accept valid file names from corpus, like speeches, bills, etc.
  if file_name not in CSV_FILENAMES:
    raise Exception("Invalid filename, must be one of the 9 files provide")
  #Only accept years belonging to a valid legislative session. (2017-2018 for all states, 2015-2016 for CA)
  if years is not None and ((not all(item > 2015 for item in years) and "CA" not in states) or (not all(item <= 2018 for item in years))):
    raise Exception("""Data for requested year not included in corpus.
     Valid session_years are 2017 and 2018 for all states provided. 2015 and 2016 are valid years for CA.""")

  payload = {}
  header_row = True

  #If no states specified, retrieve relevant files for all valid states
  if states is None:
    states = VALID_STATES

  #If no years/session specified, retrieve data for all valid state legislative session years
  if years is None:
    if "CA" in states:
      years= [2015,2016,2017,2018]
    else:
      years = [2017,2018]

  #The following code block operates as follows:
  # For every state and year requested, read the relevant CSV file(s), then
  # load it into a python object (payload) which is returned to user
  for state in states:
    FILE_PATHS = []

    #Build the filepaths to the correct data location given the states and years provided
    #Years 2017 and 2018 are valid inputs that belong to the same 2017-2018 session
    if 2017 in years or 2018 in years:
      FILE_PATHS.append(CORPUS_FILE_PATH + state + "/2017-2018/CSV/" + file_name + ".csv")

    #CA has 2 valid legislative sessions (2015-2016 and 2017-2018)
    #This means the entirety of CA data is located in more than one folder, unlike other states.
    #Looping through a list of filepaths allows us to handle this corner case
    if state == "CA" and (2015 in years or 2016 in years):
      FILE_PATHS.append(CORPUS_FILE_PATH + state + "/2015-2016/CSV/" + file_name + ".csv")

    for FILE_PATH in FILE_PATHS:
      #Open the file to read with UTF-8 encoding
      with open(FILE_PATH, newline='', encoding='utf-8', errors='replace') as csvfile:
        rows = csv.reader(csvfile, delimiter=',')
        #Read CSV row by row
        for row in rows:
          #The first row of every CSV we visit is the header row, containing the names for each column
          # We will add this to the payload only once, as every CSV we read after this will be the same headers
          if header_row:
            payload['column_headers'] = row
            #Sets up 'rows' in payload where we will store future records
            payload['rows'] = []
            header_row = False
            continue
          #Load CSV Into payload row by row
          payload['rows'].append(row)

  return payload

#Example Usage:
load_content("legislature", states= ["FL", "TX"], years=[2017])

{'column_headers': ['pid',
  'Lastname',
  'Firstname',
  'house',
  'district',
  'party',
  'biography',
  'twitter'],
 'rows': [['105013',
   'Ahern',
   'Larry',
   'House',
   '66',
   'Republican',
   'NaN',
   'NaN'],
  ['105014', 'Albritton', 'Ben', 'House', '56', 'Republican', 'NaN', 'NaN'],
  ['105015', 'Berman', 'Lori', 'House', '90', 'Democrat', 'NaN', 'NaN'],
  ['105015', 'Berman', 'Lori', 'Senate', '31', 'Democrat', 'NaN', 'NaN'],
  ['105016', 'Bileca', 'Michael', 'House', '115', 'Republican', 'NaN', 'NaN'],
  ['105017', 'Boyd', 'Jim', 'House', '71', 'Republican', 'NaN', 'NaN'],
  ['105018', 'Brodeur', 'Jason', 'House', '28', 'Republican', 'NaN', 'NaN'],
  ['105019', 'Corcoran', 'Richard', 'House', '37', 'Republican', 'NaN', 'NaN'],
  ['105020', 'Cruz', 'Janet', 'House', '62', 'Democrat', 'NaN', 'NaN'],
  ['105021', 'Diaz', 'Jose', 'House', '116', 'Republican', 'NaN', 'NaN'],
  ['105022', 'Drake', 'Brad', 'House', '5', 'Republican', 'NaN', 'NaN'],
  ['105023', 'Eisnaugle'

# Find Bills & Hearings


In [3]:
from datetime import datetime, timedelta
#This is a helper function for calculating transcript times
def add_seconds(start_time, seconds_to_add):
    # Parse the start time
    time_obj = datetime.strptime(start_time, '%H:%M:%S')
    # Add the seconds
    new_time = time_obj + timedelta(seconds=seconds_to_add)
    # Format the new time back to string
    return new_time.strftime('%H:%M:%S')

In [4]:
#Retrieve Committee Name & ID, hearing date, list of videos of the hearing, and state for a given HID
#Input:
#  Required: HID (type:Positive Int) (Ex: 10003)
#  Optional: speeches
#      -If searching through from a specific state or session, pass in
#         speeches=load_content("speeches", specific states, specific years)
#Output:
#   Dictionary
#   If no matches exist does not exist in database, returns []
def get_metadata_hearing(hid, hearings=None, videos=None):
  HID_IDX=0
  CID_IDX=4
  CNAME_IDX=8
  HDATE_IDX=1
  STATE_IDX=3

  if hearings is None:
    hearings=load_content("hearings")
  hid = str(hid)

  hearingData=None
  for row in hearings['rows']:
    if hid == row[HID_IDX]:
      hearingData={'hid':row[HID_IDX],'cid':row[CID_IDX],'cname':row[CNAME_IDX],'hearing_date':row[HDATE_IDX],'state':row[STATE_IDX]}
  if hearingData is None:
    return {}

  #vids, videos = videos_from_hid(hid)
  #hearingData['vids'] = vids
  # hearingData['videos'] = videos_from_hid(hid)

  return hearingData


#Create transcript for a given bill discussion with metadata
#Input:
#  Required: HID (type:Positive Int) (Ex: 10003)
#  Required: BID (type: String)
#  Optional: speeches
#      -If searching through from a specific state or session, pass in
#         speeches=load_content("speeches", specific states, specific years)
#Output:
#   Dictionary and Transcript
#   If no matches exist does not exist in database, returns []
def get_hearing_transcript(hid, bid, speeches=None):
  PID_IDX=1
  BID_IDX=4
  HID_IDX=3
  VID_START_IDX = 9
  VID_END_IDX = 10
  LAST_NAME_IDX = 14
  FIRST_NAME_IDX = 15
  TEXT_IDX = 16
  STARTING_TIME_IDX = 11
  if speeches is None:
    speeches=load_content("speeches")

  hid=str(hid)

  lines = []
  for row in speeches['rows']:
    if hid == row[HID_IDX] and bid == row[BID_IDX]:
      offset_time = add_seconds("00:00:00", int(row[STARTING_TIME_IDX]))
      #line = "[{}] {} {} (pid {}) speaking: {}".format(offset_time,row[FIRST_NAME_IDX],row[LAST_NAME_IDX],row[PID_IDX],row[TEXT_IDX])
      line = {'video start':row[VID_START_IDX],'video end':row[VID_END_IDX],'offset':offset_time,'bid':row[BID_IDX],
              'first name':row[FIRST_NAME_IDX],'last name':row[LAST_NAME_IDX],'pid':row[PID_IDX],'text':row[TEXT_IDX]}
      lines.append(line)
  return lines

#Helper function to get bill discussion metadata
def bill_discussion_info(hid, bid, hearings=None, speeches=None, videos=None):
  return {"metadata":get_metadata_hearing(hid, hearings,videos), "transcript":get_hearing_transcript(hid, bid, speeches)}


# accepts the output from get_hearing_transcript() and prints a transceript.
def pprint_discussion(metadata, transcript_info):
  # videos = {}
  # for vid, url in metadata['videos']:
  #   videos[vid] = url

  print()
  print(f"] Discussion of {metadata['state']} {metadata['cname']} held on {metadata['hearing_date']}")
  # print(f"] {len(videos.keys())} videos")
  print("] printing transcript: ")

  prev_video = -1
  for line in transcript_info:
    video = line['video start']
    if video != prev_video:
      print()
      print(f"] Discussing {line['bid']}")
      # print(f"] Video: {videos[line['video start']] if line['video start'] in videos else line['video start']}")
      print()
      prev_video = video
    print(f"[{line['offset']}] {line['first name']} {line['last name']}: ")
    print(f"\t{line['text']}")
  print()


# Discussion Transcripts

In [5]:
#Usage Example:
#Get metadata, videos, and recreate transcript from hearing 51835 and bill CA_201720180AB10
# discussion will contain all the information from bill CA_201720180AB10 in hearing id 51835
# 'CA_201720180AB10': ['51835',
#   '52298',
#   '52708',
#   '52856',
#   '53960',
#   '54363',
#   '54505']
discussion = bill_discussion_info(51835, "CA_201720180AB10")
discussion

{'metadata': {'hid': '51835',
  'cid': '546',
  'cname': 'Assembly Standing Committee on Education',
  'hearing_date': '2017-03-15',
  'state': 'CA'},
 'transcript': [{'video start': '24134',
   'video end': '24134',
   'offset': '00:09:25',
   'bid': 'CA_201720180AB10',
   'first name': 'Patrick',
   'last name': "O'Donnell",
   'pid': '113',
   'text': "She'll be presenting AB 10, file item number one."},
  {'video start': '24134',
   'video end': '24134',
   'offset': '00:09:29',
   'bid': 'CA_201720180AB10',
   'first name': 'Cristina',
   'last name': 'Garcia',
   'pid': '112',
   'text': "Thank you. Good afternoon Chair and members. I'm here presenting AB 10 would require vital menstrual products, pads, and tampons to be provided in school, colleges, shelters throughout the state. Menstrual products are a basic necessity similar to toilet paper. As early as the fourth grade young girls get their period and may not be prepared nor understand the natural changes of their bodies man

In [6]:
# Using pprint_discussion we layout the transcript in a human readable format
pprint_discussion(discussion['metadata'], discussion['transcript'])


] Discussion of CA Assembly Standing Committee on Education held on 2017-03-15
] printing transcript: 

] Discussing CA_201720180AB10

[00:09:25] Patrick O'Donnell: 
	She'll be presenting AB 10, file item number one.
[00:09:29] Cristina Garcia: 
	Thank you. Good afternoon Chair and members. I'm here presenting AB 10 would require vital menstrual products, pads, and tampons to be provided in school, colleges, shelters throughout the state. Menstrual products are a basic necessity similar to toilet paper. As early as the fourth grade young girls get their period and may not be prepared nor understand the natural changes of their bodies many of these young girls end up missing school three to five days a month on a regular basis, because they lack basic access to these products. No young girl should ever have to miss school because she does not have access to these basic health needs. For some young girls, access to these products is an additional barrier to their education. It's also lo